Скачивание необходимых библиотек

In [ ]:
!pip install rdkit -qq

In [ ]:
!pip install optuna -qq

In [ ]:
!pip install xgboost -qq

Импортирование библиотек

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from rdkit.Chem import PandasTools
from rdkit import DataStructs
import optuna

Чтение файла и удаление нулевых элементов

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/BigSolDB/BigSolDBv2.0.csv')
df = df.dropna()

Просмотр данных

In [ ]:
df.columns

Index(['SMILES_Solute', 'Temperature_K', 'Solvent', 'SMILES_Solvent',
       'Solubility(mole_fraction)', 'Solubility(mol/L)', 'LogS(mol/L)',
       'Compound_Name', 'CAS', 'PubChem_CID', 'FDA_Approved', 'Source'],
      dtype='object')

In [ ]:
df['Solvent'].unique()

array(['ethanol', 'methanol', 'isopropanol', 'water', 'ethyl acetate',
       'n-propanol', 'acetone', 'n-butanol', 'acetonitrile', 'DMF',
       'toluene', 'isobutanol', '1,4-dioxane', 'methyl acetate', 'THF',
       '2-butanone', 'n-pentanol', 'sec-butanol', 'n-hexane',
       'ethylene glycol', 'NMP', 'cyclohexane', 'DMSO', 'n-butyl acetate',
       'n-octanol', 'chloroform', 'n-propyl acetate', 'acetic acid',
       'dichloromethane', 'cyclohexanone', 'propylene glycol',
       'isopropyl acetate', 'DMAc', '2-ethoxyethanol', 'isopentanol',
       'n-heptane', 'ethyl formate', 'benzene', '1,2-dichloroethane',
       'n-hexanol', '2-methoxyethanol', 'isobutyl acetate',
       'tetrachloromethane', 'n-pentyl acetate', 'transcutol',
       'n-heptanol', 'ethylbenzene', 'MIBK', '2-propoxyethanol',
       'tert-butanol', 'MTBE', '2-butoxyethanol', 'propionic acid',
       'o-xylene', 'formic acid', 'diethyl ether', 'm-xylene', 'p-xylene',
       'chlorobenzene', 'dimethyl carbonate', 'n-

Создание копии исходного датасета для обработки признаков

In [ ]:
df_processed = df.copy()

Нормализация представлений молекул

In [ ]:
PandasTools.AddMoleculeColumnToFrame(
    df_processed,
    'SMILES_Solute',
    'Mol_Solute')
PandasTools.AddMoleculeColumnToFrame(
    df_processed,
    'SMILES_Solvent',
    'Mol_Solvent')
df_processed.head()

,SMILES_Solute,Temperature_K,Solvent,SMILES_Solvent,Solubility(mole_fraction),Solubility(mol/L),LogS(mol/L),Compound_Name,CAS,PubChem_CID,FDA_Approved,Source,Mol_Solute,Mol_Solvent
0,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,311.25,ethanol,CCO,0.0006,0.010083,-1.996419,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x793ec670ad50>,<rdkit.Chem.rdchem.Mol object at 0x793f092fece0>
1,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,314.65,ethanol,CCO,0.0012,0.020100,-1.696799,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x793ec670a500>,<rdkit.Chem.rdchem.Mol object at 0x793ec7c53a70>
2,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,319.15,ethanol,CCO,0.0020,0.033356,-1.476824,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x793ec670b450>,<rdkit.Chem.rdchem.Mol object at 0x793f091df300>
3,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,322.15,ethanol,CCO,0.0050,0.083356,-1.079064,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x793ec670a6c0>,<rdkit.Chem.rdchem.Mol object at 0x793f091df5a0>
4,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,324.15,ethanol,CCO,0.0139,0.233286,-0.632111,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x793ec670b220>,<rdkit.Chem.rdchem.Mol object at 0x793f091ded50>


Получение моргановских отпечатков для каждой молекулы

In [ ]:
from rdkit.Chem import AllChem

def morgan_fp(mol):
  morgan = AllChem.GetMorganGenerator(radius=2, fpSize=512)
  return morgan.GetFingerprint(mol)

In [ ]:
df_processed['Morgan_Solute'] = df_processed['Mol_Solute'].apply(morgan_fp)
df_processed['Morgan_Solvent'] = df_processed['Mol_Solvent'].apply(morgan_fp)
df_processed.head()

,SMILES_Solute,Temperature_K,Solvent,SMILES_Solvent,Solubility(mole_fraction),Solubility(mol/L),LogS(mol/L),Compound_Name,CAS,PubChem_CID,FDA_Approved,Source,Mol_Solute,Mol_Solvent,Morgan_Solute,Morgan_Solvent
0,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,311.25,ethanol,CCO,0.0006,0.010083,-1.996419,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x793ec670ad50>,<rdkit.Chem.rdchem.Mol object at 0x793f092fece0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,314.65,ethanol,CCO,0.0012,0.020100,-1.696799,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x793ec670a500>,<rdkit.Chem.rdchem.Mol object at 0x793ec7c53a70>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,319.15,ethanol,CCO,0.0020,0.033356,-1.476824,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x793ec670b450>,<rdkit.Chem.rdchem.Mol object at 0x793f091df300>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,322.15,ethanol,CCO,0.0050,0.083356,-1.079064,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x793ec670a6c0>,<rdkit.Chem.rdchem.Mol object at 0x793f091df5a0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,324.15,ethanol,CCO,0.0139,0.233286,-0.632111,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x793ec670b220>,<rdkit.Chem.rdchem.Mol object at 0x793f091ded50>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [ ]:
df_processed.columns

Index(['SMILES_Solute', 'Temperature_K', 'Solvent', 'SMILES_Solvent',
       'Solubility(mole_fraction)', 'Solubility(mol/L)', 'LogS(mol/L)',
       'Compound_Name', 'CAS', 'PubChem_CID', 'FDA_Approved', 'Source',
       'Mol_Solute', 'Mol_Solvent', 'Morgan_Solute', 'Morgan_Solvent'],
      dtype='object')

Разделение выборки на обучающую и целевую функцию

In [ ]:
X = df_processed[['Temperature_K',
                  'Morgan_Solute',
                  'Morgan_Solvent']]
y = df_processed['LogS(mol/L)']

In [ ]:
X.head()

,Temperature_K,Morgan_Solute,Morgan_Solvent
0,311.25,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,314.65,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,319.15,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,322.15,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,324.15,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [ ]:
y.head()

,LogS(mol/L)
0,-1.996419
1,-1.696799
2,-1.476824
3,-1.079064
4,-0.632111


Кодирование категориального признака методом OneHotEncoder - названия растворителя

In [ ]:
X['Morgan_Solute'] = X['Morgan_Solute'].tolist()
X['Morgan_Solvent'] = X['Morgan_Solvent'].tolist()

/tmp/ipython-input-1126101153.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Morgan_Solute'] = X['Morgan_Solute'].tolist()
/tmp/ipython-input-1126101153.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Morgan_Solvent'] = X['Morgan_Solvent'].tolist()


Приведение моргановских отпечатков типа bitvect к массиву numpy

In [ ]:
def bitvect_to_array(bitvect):
  arr = np.zeros((1,), dtype=int)
  DataStructs.ConvertToNumpyArray(bitvect, arr)
  return arr

In [ ]:
X['Morgan_Solute'] = X['Morgan_Solute'].apply(bitvect_to_array)
X['Morgan_Solvent'] = X['Morgan_Solvent'].apply(bitvect_to_array)

/tmp/ipython-input-1609987353.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Morgan_Solute'] = X['Morgan_Solute'].apply(bitvect_to_array)
/tmp/ipython-input-1609987353.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Morgan_Solvent'] = X['Morgan_Solvent'].apply(bitvect_to_array)


Вертикальное соединения массивов (по строкам)

In [ ]:
solute_fp = np.vstack(X['Morgan_Solute'].values)
solvent_fp = np.vstack(X['Morgan_Solvent'].values)

Объединение данных для получения результирующей выборки

In [ ]:
X_base = X.drop(['Morgan_Solute', 'Morgan_Solvent'], axis=1)

solute_df = pd.DataFrame(solute_fp, index=X.index).add_prefix('SoluteFP_')
solvent_df = pd.DataFrame(solvent_fp, index=X.index).add_prefix('SolventFP_')

X_final = pd.concat([X_base, solute_df, solvent_df], axis=1)

In [ ]:
X_final.shape, y.shape

((93929, 1025), (93929,))

Разделение на трейн и тест для обучения модели

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y,
    test_size=0.2,
    random_state=42
)

Подбор оптимальных гиперпараметров для модели бустинга

In [ ]:
def objective(trial):
  n_estimators = trial.suggest_int('n_estimators', 100, 1000)
  max_depth = trial.suggest_int('max_depth', 3, 10)
  learning_rate = trial.suggest_float('learning_rate', 0.01, 0.1)
  subsample = trial.suggest_float('subsample', 0.5, 1.0)
  min_child_weight = trial.suggest_int('min_child_weight', 1, 10)
  gamma = trial.suggest_float('gamma', 0, 5)

  model = xgb.XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        min_child_weight=min_child_weight,
        gamma=gamma,
        random_state=42,
        tree_method='hist',
        device='cuda'
    )
  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  rmse = mean_squared_error(y_test, y_pred)
  r2 = r2_score(y_test, y_pred)

  print(f"Trial {trial.number}: RMSE={rmse:.4f}, R2={r2:.4f}")

  return rmse

Обучение модели

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print(f"Best trial value (RMSE): {study.best_value}")
print(f"Best hyperparameters: {study.best_params}")

[I 2025-10-05 14:44:55,778] A new study created in memory with name: no-name-932f90c0-076e-4de8-9ece-6a597d591f76
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:729: UserWarning: [14:45:09] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2025-10-05 14:45:09,975] Trial 0 finished with value: 0.5147531461499818 and parameters: {'n_estimators': 294, 'max_depth': 8, 'learning_rate': 0.0140160665731306, 'subsample': 0.7010858480832709, 'min_child_weight': 9, 'gamma': 0.2076101414382553}. Best is trial 0 with value: 0.5147531461499818.


Trial 0: RMSE=0.5148, R2=0.6509


[I 2025-10-05 14:45:24,046] Trial 1 finished with value: 0.12436957423166098 and parameters: {'n_estimators': 265, 'max_depth': 10, 'learning_rate': 0.06855403158115012, 'subsample': 0.8657061759388995, 'min_child_weight': 5, 'gamma': 1.1984023111128606}. Best is trial 1 with value: 0.12436957423166098.


Trial 1: RMSE=0.1244, R2=0.9157


[I 2025-10-05 14:45:40,082] Trial 2 finished with value: 0.24221568448214756 and parameters: {'n_estimators': 971, 'max_depth': 5, 'learning_rate': 0.08550209628152534, 'subsample': 0.6397827114480941, 'min_child_weight': 2, 'gamma': 3.1031924848040466}. Best is trial 1 with value: 0.12436957423166098.


...

Trial 27: RMSE=0.2682, R2=0.8181


[I 2025-10-05 14:52:48,088] Trial 28 finished with value: 0.19497690266418746 and parameters: {'n_estimators': 364, 'max_depth': 10, 'learning_rate': 0.024289609731010617, 'subsample': 0.8037026936310561, 'min_child_weight': 2, 'gamma': 0.8486472111378109}. Best is trial 19 with value: 0.07272037597562865.


Trial 28: RMSE=0.1950, R2=0.8678


[I 2025-10-05 14:53:07,016] Trial 29 finished with value: 0.08540332304816967 and parameters: {'n_estimators': 712, 'max_depth': 8, 'learning_rate': 0.07878211671270553, 'subsample': 0.9045721130824127, 'min_child_weight': 1, 'gamma': 0.32646144530137966}. Best is trial 19 with value: 0.07272037597562865.


Trial 29: RMSE=0.0854, R2=0.9421
Best trial value (RMSE): 0.07272037597562865
Best hyperparameters: {'n_estimators': 461, 'max_depth': 10, 'learning_rate': 0.05353460973995613, 'subsample': 0.9511490911448504, 'min_child_weight': 2, 'gamma': 0.08259103510032961}


In [ ]:
xgb_model = xgb.XGBRegressor(
        n_estimators=study.best_params['n_estimators'],
        max_depth=study.best_params['max_depth'],
        learning_rate=study.best_params['learning_rate'],
        subsample=study.best_params['subsample'],
        min_child_weight=study.best_params['min_child_weight'],
        gamma=study.best_params['gamma'],
        random_state=42,
        tree_method='hist',
        device='cuda'
    )
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0.08259103510032961, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05353460973995613, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=2, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=461, n_jobs=None,
             num_parallel_tree=None, ...)

In [ ]:
from joblib import dump, load
from pathlib import Path

model_dir_path = '/content/drive/MyDrive/XGB_solubility'

dump(xgb_model, Path(model_dir_path) / 'model.joblib')

['/content/drive/MyDrive/XGB_solubility/xgb_for_solubility_pred.joblib']